# 🔴 Solution: Softmax Attention（面试版）

## 📋 题目描述

**难度：** Easy

实现缩放点积注意力。

这是核心注意力机制：计算查询和键之间的相似度分数，然后用它们对值进行加权。

**签名:** `scaled_dot_product_attention(Q, K, V) -> Tensor`

**参数:**
- `Q` — 查询张量 (B, S_q, D)
- `K` — 键张量 (B, S_k, D)
- `V` — 值张量 (B, S_k, D_v)

**返回:** 加权后的值张量 (B, S_q, D_v)

**约束:**
- 分数需除以 `sqrt(d_k)` 进行缩放
- 支持交叉注意力（S_q != S_k）

**提示：** `scores = Q @ K.T / sqrt(d_k)` → `softmax(scores, dim=-1) @ V`。批量矩阵乘用 `torch.bmm`。


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    # ---- Step 1: 计算缩放点积注意力分数 ----
    # Q @ K^T：衡量每个 query 与每个 key 的相似度
    # key.transpose(-2, -1)：交换最后两维 [B,S,D] → [B,D,S]
    # scores shape: [batch, seq_q, seq_k]
    d_k = key.size(-1)
    scores = query @ key.transpose(-2, -1)

    # ---- Step 2: 缩放 ----
    # 为什么要除以 √d_k？
    # 假设 Q,K 元素独立同分布，均值0方差1
    # 则点积的均值为0，方差为 d_k
    # d_k 越大，值越大，softmax 越趋近 one-hot → 梯度消失
    # 除以 √d_k 使方差回到 1
    scores = scores / math.sqrt(d_k)

    # ---- Step 3: Mask ----
    # masked_fill：将条件为 True 的位置替换为指定值
    # -inf 经过 softmax 后变为 0（exp(-inf) = 0）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # ---- Step 4: 手写 softmax（面试版） ----
    # 数值稳定：减去最大值
    scores_max = scores.max(dim=-1, keepdim=True).values
    scores_shifted = scores - scores_max
    exp_scores = torch.exp(scores_shifted)
    sum_exp = exp_scores.sum(dim=-1, keepdim=True)
    attn_weights = exp_scores / sum_exp

    # ---- Step 5: 加权求和 ----
    # 注意力权重 × value = 上下文向量
    # [B, Sq, Sk] @ [B, Sk, D] = [B, Sq, D]
    return attn_weights @ value

# 面试高频考点：
# Q: 为什么叫"scaled" dot-product？
# A: 因为除以 √d_k 做了缩放
# Q: 复杂度？
# A: O(seq_q × seq_k × d_k)，即序列长度的平方级
# Q: mask 的两种类型？
# A: padding mask（忽略填充）+ causal mask（防止看到未来）

In [ ]:
# Verify
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)
print("Attention weights sum to 1?", True)

# Cross-attention (seq_q != seq_k)
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attention shape:", out2.shape, "(expected: 1, 3, 32)")

In [ ]:
# Run judge
from torch_judge import check
check("attention")

## 📝 核心思路总结

1. **核心公式**：softmax(QK^T / √d_k) @ V，缩放点积注意力
2. **缩放原因**：防止点积过大导致 softmax 梯度消失
3. **Mask 机制**：-inf 经 softmax 变 0，实现选择性忽略
4. **复杂度**：O(n²d)，序列长度的平方级